# 心電図のピーク検出を作る

読み込みとグラフは用意してあります。**山（R）を見つける関数**を、自分で書いてください。

- セルを実行する: `Shift + Enter`（上から順に実行します）
- `# ここに書く` のところを書きかえます
- データは**合成データ**です（実際の人の計測ではありません）

In [ ]:
import os

import matplotlib.pyplot as plt

DATA_URL = "https://misaki113.github.io/kamiyama-signal/data/"  # Colab などデータが手元にない環境では、配布ページの URL + "data/" を入れる


def data_path(name):
    """データファイルの場所を探す。見つからなければ DATA_URL から取ってくる。"""
    for folder in ["data", "../data", "."]:
        path = os.path.join(folder, name)
        if os.path.exists(path):
            return path
    if DATA_URL != "":
        import urllib.request

        urllib.request.urlretrieve(DATA_URL + name, name)
        return name
    raise FileNotFoundError(name + " が見つかりません。data フォルダの場所を確かめてください")


def load_ecg(name):
    """CSV を読んで、サンプリング周波数 fs、時刻の配列 t、心電図の配列 ecg を返す。"""
    fs = 0.0
    ecg = []
    header_done = False
    with open(data_path(name)) as f:
        for line in f:
            line = line.strip()
            if line.startswith("#"):  # メタデータの行
                key, value = line[1:].split(":", 1)
                if key.strip() == "sampling_rate_hz":
                    fs = float(value)
            elif not header_done:  # 列名の行
                header_done = True
            else:  # データの行
                ecg.append(float(line.split(",")[1]))
    t = [i / fs for i in range(len(ecg))]
    return fs, t, ecg


def show(t, x, peaks=None, threshold=None, t_from=2.0, t_to=8.0, title=""):
    """波形を描く。peaks（配列の番号のリスト）を渡すと、その位置に印を付ける。"""
    plt.figure(figsize=(11, 3))
    plt.plot(t, x, linewidth=1)
    if threshold is not None:
        plt.axhline(threshold, color="orange", linestyle="--")
    if peaks:
        plt.plot([t[i] for i in peaks], [x[i] for i in peaks], "rv")
    plt.xlim(t_from, t_to)
    plt.xlabel("time [s]")
    plt.ylabel("[mV]")
    plt.title(title)
    plt.show()


fs, t, ecg = load_ecg("ecg_clean.csv")
print("fs =", fs, " 点の数 =", len(ecg))
show(t, ecg, title="ecg_clean.csv")

## 1. しきい値を超えた点を拾う

いちばん素朴な方法です。`x[i]` がしきい値より大きい `i` を、すべて `peaks` に入れます。

In [ ]:
def detect_v1(x, threshold):
    peaks = []
    # ここに書く
    return peaks


threshold = 0.5 * max(ecg)  # しきい値は、最大値の半分にしてみる
peaks = detect_v1(ecg, threshold)
print("見つかった点の数:", len(peaks))
show(t, ecg, peaks, threshold, title="v1")

このデータは 30 秒間で、心臓は 34 回打っています。見つかった点の数は、いくつでしたか。

多すぎるときは、1 つの山を拡大して見てください（`t_from=2.0, t_to=2.1` など）。

In [ ]:
show(t, ecg, peaks, threshold, t_from=2.0, t_to=2.1, title="v1 (zoom)")

## 2. 山 1 つにつき、てっぺんを 1 点だけ選ぶ

考え方は 2 つあります。どちらでもかまいません。

- しきい値を超えてから下回るまでを 1 つの山と考えて、その中でいちばん高い点を選ぶ
- となりとの差が ＋ から − に変わる点（`x[i-1] < x[i]` かつ `x[i] >= x[i+1]`）を選ぶ

In [ ]:
def detect_v2(x, threshold):
    peaks = []
    # ここに書く
    return peaks


peaks = detect_v2(ecg, threshold)
print("見つかった山の数:", len(peaks), "（34 になれば正解）")
show(t, ecg, peaks, threshold, title="v2")

## 3. 心拍数を出す

となり合う山の間隔（秒）が、1 拍にかかる時間です。心拍数は `60 ÷ 間隔` [回／分]。
`peaks` に入っているのは配列の番号なので、秒に直すには `fs` で割ります。

In [ ]:
def heart_rates(peaks, fs):
    bpm = []
    # ここに書く
    return bpm


bpm = heart_rates(peaks, fs)
if bpm:
    print("平均の心拍数:", sum(bpm) / len(bpm), "回／分（約 70 になれば正解）")

## 4. ノイズのあるデータで試す

同じ関数を `ecg_noisy.csv` に使います。山の数は 34 になりますか。
ならないときは、どこで何が起きているかを、拡大して確かめてください。

In [ ]:
fs, t, noisy = load_ecg("ecg_noisy.csv")
threshold = 0.5 * max(noisy)
peaks = detect_v2(noisy, threshold)
print("見つかった山の数:", len(peaks))
show(t, noisy, peaks, threshold, title="ecg_noisy.csv, v2")

## 5. 打った直後は探さない（不応期）

心臓は、1 回打った直後にはもう 1 回打てません。
「直前の山から 200 ms 以内に見つかった山は、同じ拍の一部」とみなして、高い方だけを残します。
200 ms は、配列の番号でいくつ分でしょうか。

In [ ]:
def detect_v3(x, threshold, fs):
    peaks = []
    # ここに書く（detect_v2 の結果を使うと楽です）
    return peaks


peaks = detect_v3(noisy, threshold, fs)
print("見つかった山の数:", len(peaks), "（34 になれば正解）")
show(t, noisy, peaks, threshold, title="ecg_noisy.csv, v3")

## 6. 挑戦：全体が揺れるデータ

`ecg_hard.csv` は、全体がゆっくり上下に揺れています。ここまでの関数で、山は 34 個になりますか。
ならないとき、直すのは検出の手順でしょうか。それとも、その前のデータの整え方でしょうか。

In [ ]:
fs, t, hard = load_ecg("ecg_hard.csv")
threshold = 0.5 * max(hard)
peaks = detect_v3(hard, threshold, fs)
print("見つかった山の数:", len(peaks))
show(t, hard, peaks, threshold, t_from=9.0, t_to=21.0, title="ecg_hard.csv, v3")

In [ ]:
# ここに、自分の工夫を書く